# Verificación de la capa Bronze — Iceberg

Revisamos las tablas Iceberg en `nessie.nyc` tras la ingesta:
referencias Nessie, metadatos, snapshots, particionado y calidad de datos.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("verify-bronze-iceberg")
spark.version

26/05/31 16:51:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

La ingesta bronze usa WAP: cada mes crea una rama efímera, escribe, valida y
hace merge a `bronze`. Tras el último mes de cada año se crea un tag de auditoría.

In [2]:
# Referencias Nessie (ramas + tags)
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:

  * Branch bronze                                   6c92d3f09112
  * Branch main                                     e350bde574fc
  * Branch silver                                   9aac1b2ec1a5
    Tag    bronze-taxi_zone_lookup-2026             6c92d3f09112
    Tag    bronze-yellow_taxi-2023                  0ffe14008656
    Tag    bronze-yellow_taxi-2024                  fb478efd4d86
    Tag    bronze-yellow_taxi-2025                  ee10bf94a0f6

  Branches: 3  |  Tags: 4


In [3]:
# Cambiamos a la rama bronze para el resto del notebook
nessie.set_spark_branch(spark, "bronze")

# Namespaces disponibles
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

+---------+
|namespace|
+---------+
|nyc      |
+---------+



In [4]:
# Tablas en el namespace bronze
spark.sql("SHOW TABLES IN nessie.nyc").show(truncate=False)

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|nyc      |taxi_zone_lookup|false      |
|nyc      |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [5]:
# Esquema, particionado, propiedades
spark.sql("DESCRIBE EXTENDED nessie.nyc.yellow_taxi").show(250, truncate=False)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
# Propiedades Iceberg (versión de formato, gc, etc.)
spark.sql("SHOW TBLPROPERTIES nessie.nyc.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |1757265477960251573                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |6c92d3f0911233519ce561af4814d7716b7978c5d14e1ed2a1cc404211d55511|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [7]:
# Cada write/overwrite crea un snapshot
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                      

In [8]:
# Historial (registro de snapshots ordenado por tiempo)
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-31 16:38:57.482|751264134898552722 |NULL               |true               |
|2026-05-31 16:39:15.652|179461454785393106 |751264134898552722 |true               |
|2026-05-31 16:39:34.254|8663488671414409298|179461454785393106 |true               |
|2026-05-31 16:39:47.106|3042831213633643212|8663488671414409298|true               |
|2026-05-31 16:40:01.572|8983329294462776866|3042831213633643212|true               |
|2026-05-31 16:40:16.811|1001402227516272329|8983329294462776866|true               |
|2026-05-31 16:40:31.102|3423163687281835035|1001402227516272329|true               |
|2026-05-31 16:40:45.461|6547510629800845180|3423163687281835035|true               |
|2026-05-31 16:41:03.05 |1757265477960251573|654751062

## 3b. Crecimiento de registros por tag

Cada tag Nessie captura el estado de `bronze` tras ingestar todos los meses
de un año. Leyendo la tabla en cada tag vemos cómo crece año a año.

In [9]:
# Conteo de registros en cada tag anual para ver el crecimiento incremental
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

tag_rows_df = refs_df.filter(refs_df[type_col].ilike("tag")) \
    .filter(refs_df["name"].rlike(r"^bronze-yellow_taxi-\d{4}$"))
tag_names = [r["name"] for r in tag_rows_df.collect()]

tag_pattern = re.compile(r"bronze-yellow_taxi-(\d{4})")
tag_rows = []
for name in tag_names:
    m = tag_pattern.match(name)
    if m:
        tag_rows.append((name, int(m.group(1))))
tag_rows.sort(key=lambda t: t[1])

growth = []
for tag_name, year in tag_rows:
    nessie.set_spark_branch(spark, tag_name)
    count = spark.table("nessie.nyc.yellow_taxi").count()
    growth.append((tag_name, year, count))

nessie.set_spark_branch(spark, "bronze")

growth_df = spark.createDataFrame(
    [(tag, year, total, total - (growth[i-1][2] if i > 0 else 0))
     for i, (tag, year, total) in enumerate(growth)],
    ["tag", "year", "cumulative_records", "records_added"]
)
growth_df.show(truncate=False)

+-----------------------+----+------------------+-------------+
|tag                    |year|cumulative_records|records_added|
+-----------------------+----+------------------+-------------+
|bronze-yellow_taxi-2023|2023|9384487           |9384487      |
|bronze-yellow_taxi-2024|2024|18939265          |9554778      |
|bronze-yellow_taxi-2025|2025|30137291          |11198026     |
+-----------------------+----+------------------+-------------+



## 4. Particionado y ficheros

In [10]:
# Especificación de particionado
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{648}    |0      |2964617     |2         |46951687                     |0                           |0                         |0                           |0                         |2026-05-31 16:40:01.572|8983329294462776866     |
|{649}    |0      |3007533     |3         |46728056         

In [11]:
# Ficheros de datos: rutas, tamaños, registros por fichero
files_df = spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format",
    "record_count", "file_size_in_bytes"
).show(100, truncate=80)

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     4145224|          65386491|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2007-1...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|           2|              6376|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2009-0...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tp

In [12]:
# Resumen: ficheros totales, tamaño, registros
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

+-----------+-------------+-------------+
|total_files|total_size_mb|total_records|
+-----------+-------------+-------------+
|         43|        451.2|     30137291|
+-----------+-------------+-------------+



## 5. Manifiestos

In [13]:
# Una entrada por snapshot
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.manifests").show(truncate=80)

+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|content|                                                                            path|length|partition_spec_id|  added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|               partition_summaries|
+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|      0|s3a://warehouse/

## 6. Calidad de datos

In [14]:
df = spark.table("nessie.nyc.yellow_taxi")
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Total records: 30,137,291
Columns: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



> **Nota** — Bronze es fiel al origen. Las celdas siguientes muestran valores
> que *parecen* anómalos pero son esperados en esta capa:
>
> - **Fechas fuera de ventana** (filas en 2001, 2008, 2014, 2022, meses sueltos…):
>   la ventana de carga solo restringe *qué ficheros* se descargan (Q1 de
>   2023/2024/2025), no las filas dentro de ellos. Los parquet de
>   NYC TLC contienen filas con `tpep_pickup_datetime` erróneo.
> - **Importes negativos o extremos** (`fare_amount`, `trip_distance`,
>   `tip_amount`, `total_amount`): errores de registro presentes en el origen.
>
> Bronze los conserva tal cual; el `clean()` de silver (filtrado de validez +
> outliers IQR) es quien los elimina. Ver `preprocess.ipynb` §1.3.

In [15]:
# Registros por año-mes (valida el particionado)
from pyspark.sql.functions import year, month

df.groupBy(
    year("tpep_pickup_datetime").alias("year"),
    month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2001|    1|      3|
|2002|   12|      6|
|2003|    1|      3|
|2007|   12|      1|
|2008|   12|      9|
|2009|    1|      6|
|2014|   11|      1|
|2022|   10|     11|
|2022|   12|     25|
|2023|    1|3066726|
|2023|    2|2914003|
|2023|    3|3403619|
|2023|    4|     85|
|2023|   12|     10|
|2024|    1|2964617|
|2024|    2|3007533|
|2024|    3|3582607|
|2024|    4|      2|
|2024|   12|     21|
|2025|    1|3475234|
|2025|    2|3577542|
|2025|    3|4145225|
|2025|    4|      2|
+----+-----+-------+



In [16]:
# Nulos por columna
from pyspark.sql.functions import col as c, sum as spark_sum

null_counts = df.select([
    spark_sum(c(col).isNull().cast("int")).alias(col)
    for col in df.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 3251890  
 trip_distance         | 0        
 RatecodeID            | 3251890  
 store_and_fwd_flag    | 3251890  
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 3251890  
 airport_fee           | 3251890  
 cbd_congestion_fee    | 18939265 



In [17]:
# Estadísticas básicas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "passenger_count"
).summary("count", "min", "max", "mean").show()

+-------+-----------------+-----------------+------------------+------------------+------------------+
|summary|    trip_distance|      fare_amount|        tip_amount|      total_amount|   passenger_count|
+-------+-----------------+-----------------+------------------+------------------+------------------+
|  count|         30137291|         30137291|          30137291|          30137291|          26885401|
|    min|              0.0|          -1807.6|            -300.0|          -1832.85|               0.0|
|    max|        335004.33|        863372.12|            999.99|         863380.37|               9.0|
|   mean|4.784036447735362|17.98249326888429|3.1604352806629623|26.545426059993545|1.3263808488480422|
+-------+-----------------+-----------------+------------------+------------------+------------------+



In [18]:
# Filas de muestra
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-03-01 00:17:16 |2025-03-01 00:25:52  |1.0            |0.9          |1.0       |N                 |140         |236 

## 7. Taxi Zone Lookup

In [19]:
# Comprobamos si la tabla de referencia existe y la inspeccionamos
try:
    zones = spark.table("nessie.nyc.taxi_zone_lookup")
    print(f"Records: {zones.count()}")
    zones.printSchema()
    zones.show(10, truncate=False)
except Exception as e:
    print(f"Table not yet ingested: {e}")

Records: 265
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |B